<a href="https://colab.research.google.com/github/dchav12/New-stable-/blob/main/NBA_STABLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
SPORT = "basketball_nba"
SIMS = 50000

# NBA variance
SD_MARGIN = 12.0
SD_TOTAL  = 15.0

UNIT_MIN = 0.25
UNIT_CAP = 1.0
MAX_JUICE = -200

print("✅ NBA Engine Settings Loaded")

✅ NBA Engine Settings Loaded


In [ ]:
DAYS_BACK = 30

In [ ]:
# NBA Possessions
games_hist["home_poss"] = (
    games_hist["home_fga"]
    - games_hist["home_oreb"]
    + games_hist["home_tov"]
    + 0.44 * games_hist["home_fta"]
)

NameError: name 'games_hist' is not defined

In [ ]:
import os, requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

ODDS_API_KEY = os.getenv("ODDS_API_KEY")  # if set in Colab secrets
# If not set, paste your key here:
# ODDS_API_KEY = "PASTE_KEY"

SLATE_DATE = "2026-03-01"   # change as needed
SPORT = "basketball_nba"
print("✅ Loaded:", SPORT, SLATE_DATE, "key?", bool(ODDS_API_KEY))

✅ Loaded: basketball_nba 2026-03-01 key? False


In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
print("✅ ODDS_API_KEY set?", bool(os.getenv("ODDS_API_KEY")))

✅ ODDS_API_KEY set? True


In [ ]:
import os, requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

# pulls from env (set via Secrets or the one-liner cell)
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-01"   # change if needed

if not ODDS_API_KEY:
    raise ValueError("ODDS_API_KEY not found. Set it in env or Colab Secrets.")

print("✅ Ready:", SPORT, SLATE_DATE)

✅ Ready: basketball_nba 2026-03-01


In [ ]:
def pull_market_data(
    sport: str,
    date: str,
    regions="us",
    markets="h2h,spreads,totals",
    odds_format="american",
    date_filter=True,
):
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
        "dateFormat": "iso",
    }

    if date_filter:
        day = datetime.strptime(date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        params["commenceTimeFrom"] = day.isoformat().replace("+00:00", "Z")
        params["commenceTimeTo"] = (day + timedelta(days=1)).isoformat().replace("+00:00", "Z")

    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise Exception(r.text)

    data = r.json()
    rows = []

    for g in data:
        home = g.get("home_team")
        away = g.get("away_team")
        t = g.get("commence_time")

        bks = g.get("bookmakers", [])
        book = bks[0] if bks else None

        h2h = spreads = totals = None
        if book:
            for m in book.get("markets", []):
                if m.get("key") == "h2h":
                    h2h = m
                elif m.get("key") == "spreads":
                    spreads = m
                elif m.get("key") == "totals":
                    totals = m

        def pick_outcome(mkt, name):
            if not mkt: return None
            for o in mkt.get("outcomes", []):
                if o.get("name") == name:
                    return o
            return None

        home_ml = away_ml = np.nan
        if h2h:
            oh = pick_outcome(h2h, home)
            oa = pick_outcome(h2h, away)
            if oh: home_ml = oh.get("price", np.nan)
            if oa: away_ml = oa.get("price", np.nan)

        spread_home_line = spread_home_odds = spread_away_line = spread_away_odds = np.nan
        if spreads:
            oh = pick_outcome(spreads, home)
            oa = pick_outcome(spreads, away)
            if oh:
                spread_home_line = oh.get("point", np.nan)
                spread_home_odds = oh.get("price", np.nan)
            if oa:
                spread_away_line = oa.get("point", np.nan)
                spread_away_odds = oa.get("price", np.nan)

        total_line = over_odds = under_odds = np.nan
        if totals:
            oo = pick_outcome(totals, "Over")
            ou = pick_outcome(totals, "Under")
            if oo:
                total_line = oo.get("point", np.nan)
                over_odds = oo.get("price", np.nan)
            if ou:
                under_odds = ou.get("price", np.nan)

        rows.append({
            "home_team": home,
            "away_team": away,
            "commence_time": t,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "spread_home_line": spread_home_line,
            "spread_home_odds": spread_home_odds,
            "spread_away_line": spread_away_line,
            "spread_away_odds": spread_away_odds,
            "total_line": total_line,
            "over_odds": over_odds,
            "under_odds": under_odds,
        })

    df = pd.DataFrame(rows)
    print(f"✅ pulled {len(df)} NBA games for {date}")
    print("ML non-null:", int(df["home_ml"].notna().sum()), "/", len(df))
    print("Spread non-null:", int(df["spread_home_line"].notna().sum()), "/", len(df))
    print("Total non-null:", int(df["total_line"].notna().sum()), "/", len(df))
    return df

games_df = pull_market_data(SPORT, SLATE_DATE, date_filter=True)
games_df.head()

✅ pulled 8 NBA games for 2026-03-01
ML non-null: 8 / 8
Spread non-null: 8 / 8
Total non-null: 8 / 8


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds
0,New York Knicks,San Antonio Spurs,2026-03-01T18:10:00Z,108,-126,2.0,-112,-2.0,-108,227.5,-114,-106
1,Brooklyn Nets,Cleveland Cavaliers,2026-03-01T20:40:00Z,420,-560,11.0,-106,-11.0,-114,222.5,-110,-110
2,Chicago Bulls,Milwaukee Bucks,2026-03-01T20:40:00Z,136,-162,3.5,-110,-3.5,-110,228.5,-112,-108
3,Denver Nuggets,Minnesota Timberwolves,2026-03-01T20:40:00Z,-148,126,-2.5,-114,2.5,-106,237.5,-114,-106
4,Indiana Pacers,Memphis Grizzlies,2026-03-01T22:10:00Z,-102,-116,1.0,-112,-1.0,-108,238.5,-106,-114


In [ ]:
def pull_completed_scores(sport: str, days_from: int):
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/scores/"
    params = {"apiKey": ODDS_API_KEY, "daysFrom": int(days_from)}
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise Exception(r.text)

    data = r.json()
    rows = []
    for g in data:
        if not g.get("completed"):
            continue
        home = g.get("home_team")
        away = g.get("away_team")
        ct = g.get("commence_time")
        scores = {s["name"]: s["score"] for s in g.get("scores", [])} if g.get("scores") else {}
        hs = scores.get(home)
        as_ = scores.get(away)
        if hs is None or as_ is None:
            continue
        rows.append({
            "home_team": home,
            "away_team": away,
            "home_score": float(hs),
            "away_score": float(as_),
            "date": ct
        })

    return pd.DataFrame(rows)

# Pull multiple windows (stitch together)
pieces = []
for d in [3, 6, 10, 14, 21, 28]:
    try:
        df_part = pull_completed_scores(SPORT, d)
        pieces.append(df_part)
        print(f"daysFrom {d}: {len(df_part)} games")
    except Exception as e:
        print("⚠️", e)

historical_df = pd.concat(pieces, ignore_index=True).drop_duplicates(
    subset=["home_team","away_team","home_score","away_score","date"]
)

print("✅ Unique historical games pulled:", len(historical_df))
historical_df.head()

daysFrom 3: 20 games
⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

✅ Unique historical games pulled: 20


,home_team,away_team,home_score,away_score,date
0,Indiana Pacers,Charlotte Hornets,109.0,133.0,2026-02-27T00:10:00Z
1,Philadelphia 76ers,Miami Heat,124.0,117.0,2026-02-27T00:11:00Z
2,Atlanta Hawks,Washington Wizards,126.0,96.0,2026-02-27T00:41:00Z
3,Brooklyn Nets,San Antonio Spurs,110.0,126.0,2026-02-27T00:41:00Z
4,Orlando Magic,Houston Rockets,108.0,113.0,2026-02-27T00:43:00Z


In [ ]:
PPP_LEAGUE = 1.13
HOME_EDGE_PTS = 2.0

hist = historical_df.copy()
hist["total_pts"] = hist["home_score"] + hist["away_score"]

# Possession proxy
hist["poss_proxy"] = hist["total_pts"] / PPP_LEAGUE

home_rows = pd.DataFrame({
    "team": hist["home_team"],
    "points_scored": hist["home_score"],
    "points_allowed": hist["away_score"],
    "poss": hist["poss_proxy"],
})

away_rows = pd.DataFrame({
    "team": hist["away_team"],
    "points_scored": hist["away_score"],
    "points_allowed": hist["home_score"],
    "poss": hist["poss_proxy"],
})

all_stats = pd.concat([home_rows, away_rows], ignore_index=True)

team_overall = all_stats.groupby("team").agg(
    poss=("poss","mean"),
    ppp_for=("points_scored", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean()),
    ppp_against=("points_allowed", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean())
).reset_index()

team_overall = team_overall.fillna(team_overall.mean(numeric_only=True))

print("✅ Team baseline built:", team_overall.shape)
team_overall.head()

✅ Team baseline built: (30, 4)


,team,poss,ppp_for,ppp_against
0,Atlanta Hawks,196.460177,0.641351,0.488649
1,Boston Celtics,229.203540,0.645714,0.484286
2,Brooklyn Nets,219.026549,0.504505,0.625495
3,Charlotte Hornets,196.460177,0.615901,0.514099
4,Chicago Bulls,206.194690,0.543176,0.586824


In [ ]:
g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = g[c].fillna(g[c].mean())

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_points_proj"] - g["away_points_proj"] + HOME_EDGE_PTS
g["mean_total_model"] = g["home_points_proj"] + g["away_points_proj"]

games_df = g

print("Margin model range:", float(g["mean_margin_model"].min()), "to", float(g["mean_margin_model"].max()))
print("Total model range:", float(g["mean_total_model"].min()), "to", float(g["mean_total_model"].max()))

Margin model range: -19.44733660543507 to 18.616076421248835
Total model range: 215.0 to 250.0


In [ ]:
BLEND_MODEL = 0.4
BLEND_MARKET = 0.6

games_df["market_margin"] = -games_df["spread_home_line"]

games_df["mean_margin"] = (
    BLEND_MODEL * games_df["mean_margin_model"] +
    BLEND_MARKET * games_df["market_margin"]
)

games_df["mean_total"] = (
    BLEND_MODEL * games_df["mean_total_model"] +
    BLEND_MARKET * games_df["total_line"]
)

print("✅ NBA blend applied")
print("Blended margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("Blended total range:", float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

✅ NBA blend applied
Blended margin range: -10.422312754096987 to 11.890574985180791
Blended total range: 225.3 to 237.3


In [ ]:
import numpy as np
import pandas as pd

# =========================
# NBA SIM SETTINGS
# =========================
SIMS = 50000
SD_MARGIN = 12.0
SD_TOTAL  = 15.0

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# ensure numeric
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# BUILD CARDS
# =========================
# ML (home side only for simplicity)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev","commence_time"]
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# =========================
# CORRELATION CAP: max 2 per game
# =========================
kept = []
counts = {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

print("✅ Plays after cap:", len(card))
top = card.head(20).copy()

top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_card.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

✅ Plays after cap: 16
                                                                                                      discord_text
                     Philadelphia 76ers at Boston Celtics\nOVER 221.5 (-106) — 1.0u\nSim Win%: 77.8% | EV: 51.2%\n
               Minnesota Timberwolves at Denver Nuggets\nUNDER 237.5 (-106) — 0.88u\nSim Win%: 72.9% | EV: 41.7%\n
         Memphis Grizzlies at Indiana Pacers\nMemphis Grizzlies -1.0 (-108) — 0.88u\nSim Win%: 73.1% | EV: 40.8%\n
                  Milwaukee Bucks at Chicago Bulls\nChicago Bulls ML (+136) — 0.58u\nSim Win%: 59.0% | EV: 39.3%\n
                    Cleveland Cavaliers at Brooklyn Nets\nOVER 222.5 (-110) — 0.83u\nSim Win%: 72.1% | EV: 37.6%\n
                 Milwaukee Bucks at Chicago Bulls\nChicago Bulls 3.5 (-110) — 0.74u\nSim Win%: 70.0% | EV: 33.6%\n
                Portland Trail Blazers at Atlanta Hawks\nUNDER 237.5 (-106) — 0.69u\nSim Win%: 68.3% | EV: 32.7%\n
         Portland Trail Blazers at Atlanta Hawks\nAtlanta 

In [ ]:
SD_MARGIN = 14.5
SD_TOTAL  = 18.0
print("✅ Using widened NBA variance:", SD_MARGIN, SD_TOTAL)

✅ Using widened NBA variance: 14.5 18.0


In [ ]:
import numpy as np
import pandas as pd

SIMS = 50000
UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ML (home)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig
ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])
sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"
sp_card["p_win"] = sp_card["p_win"]

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])
to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"
to_card["p_win"] = to_card["p_win"]

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev","commence_time"]
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# cap: max 2 plays per game
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

print("✅ Plays after cap:", len(card))
top = card.head(20).copy()

top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_card_widevar.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

✅ Plays after cap: 15
                                                                                                      discord_text
                    Philadelphia 76ers at Boston Celtics\nOVER 221.5 (-106) — 0.93u\nSim Win%: 73.9% | EV: 43.7%\n
                  Milwaukee Bucks at Chicago Bulls\nChicago Bulls ML (+136) — 0.52u\nSim Win%: 57.4% | EV: 35.4%\n
               Minnesota Timberwolves at Denver Nuggets\nUNDER 237.5 (-106) — 0.74u\nSim Win%: 69.5% | EV: 35.1%\n
         Memphis Grizzlies at Indiana Pacers\nMemphis Grizzlies -1.0 (-108) — 0.73u\nSim Win%: 69.5% | EV: 33.9%\n
                    Cleveland Cavaliers at Brooklyn Nets\nOVER 222.5 (-110) — 0.69u\nSim Win%: 68.7% | EV: 31.2%\n
                  Milwaukee Bucks at Chicago Bulls\nChicago Bulls 3.5 (-110) — 0.6u\nSim Win%: 66.7% | EV: 27.3%\n
                Portland Trail Blazers at Atlanta Hawks\nUNDER 237.5 (-106) — 0.58u\nSim Win%: 65.5% | EV: 27.2%\n
              Cleveland Cavaliers at Brooklyn Nets\nBrookl

In [ ]:
BLEND_MODEL = 0.25
BLEND_MARKET = 0.75

games_df["market_margin"] = -pd.to_numeric(games_df["spread_home_line"], errors="coerce")
games_df["mean_margin"] = (
    BLEND_MODEL * pd.to_numeric(games_df["mean_margin_model"], errors="coerce") +
    BLEND_MARKET * games_df["market_margin"]
)

games_df["mean_total"] = (
    BLEND_MODEL * pd.to_numeric(games_df["mean_total_model"], errors="coerce") +
    BLEND_MARKET * pd.to_numeric(games_df["total_line"], errors="coerce")
)

print("✅ Strong NBA blend applied (25/75)")
print("Blended margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("Blended total range:", float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

✅ Strong NBA blend applied (25/75)
Blended margin range: -10.638945471310617 to 10.806609365737994
Blended total range: 223.875 to 237.75


In [ ]:
# ======================================
# BACK-TO-BACK DETECTION
# ======================================

from datetime import datetime, timedelta

# pull yesterday’s completed games
yesterday = (datetime.strptime(SLATE_DATE, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")

recent_games = historical_df.copy()
recent_games["date"] = pd.to_datetime(recent_games["date"]).dt.date

yesterday_date = datetime.strptime(yesterday, "%Y-%m-%d").date()

teams_played_yesterday = set(
    recent_games.loc[recent_games["date"] == yesterday_date, "home_team"]
).union(
    set(recent_games.loc[recent_games["date"] == yesterday_date, "away_team"])
)

games_df["home_b2b"] = games_df["home_team"].isin(teams_played_yesterday)
games_df["away_b2b"] = games_df["away_team"].isin(teams_played_yesterday)

# apply fatigue penalty
B2B_PENALTY = 1.5

games_df["mean_margin"] = games_df["mean_margin"] \
    - games_df["home_b2b"] * B2B_PENALTY \
    + games_df["away_b2b"] * B2B_PENALTY

print("✅ B2B adjustment applied")
print("Home B2B:", games_df["home_b2b"].sum())
print("Away B2B:", games_df["away_b2b"].sum())

✅ B2B adjustment applied
Home B2B: 4
Away B2B: 5


In [ ]:
# ======================================
# BLOWOUT ADJUSTMENT
# ======================================

BLOWOUT_THRESHOLD = 8

large_spread = games_df["spread_home_line"].abs() >= BLOWOUT_THRESHOLD

# reduce margin magnitude slightly
games_df.loc[large_spread, "mean_margin"] *= 0.90

# reduce totals slightly (garbage time compression)
games_df.loc[large_spread, "mean_total"] -= 1.5

print("✅ Blowout adjustment applied")
print("Large spread games:", large_spread.sum())

✅ Blowout adjustment applied
Large spread games: 2


In [ ]:
Run NBA Stable Engine — STRICT

Settings:
• Sims: 50,000
• Variance: SD_MARGIN=14.5 | SD_TOTAL=18.0
• Market Blend: 25% Model / 75% Market
• B2B Adjustment: ON (−1.5 pts fatigue)
• Blowout Compression: ON (≥8 spread reduce margin 10%, total −1.5)
• Correlation Filter: Max 2 plays per game
• Exposure Rule: No ML + Spread on same team unless both ≥6% EV
• Unit Cap: 1.0u
• Min Unit: 0.25u
• Max Juice: −200
• Remove 0 EV plays

Process:
1) Use blended mean_margin + mean_total
2) Run Monte Carlo simulation (50k)
3) Calculate win probability vs market implied
4) Compute EV
5) Apply Kelly sizing (capped 1u)
6) Remove correlated exposure >2 per game
7) Sort by EV descending
8) Export top 20 to CSV
9) Generate Discord-ready output

Output:
• matchup
• bet
• units
• Sim Win%
• EV%

Date: 2026-03-01

SyntaxError: invalid character '—' (U+2014) (1502304135.py, line 1)

In [ ]:
import numpy as np
import pandas as pd

# =========================
# FINAL NBA RUN CELL
# (uses current games_df with all adjustments already applied)
# =========================

SIMS = 50000

# If you already set these earlier, leave them; otherwise these are your calibrated defaults
SD_MARGIN = globals().get("SD_MARGIN", 14.5)
SD_TOTAL  = globals().get("SD_TOTAL", 18.0)

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# ensure numeric
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half-Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# BUILD MARKET CARDS
# =========================
# ML (home only)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"
sp_card["p_win"] = sp_card["p_win"]

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"
to_card["p_win"] = to_card["p_win"]

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev","commence_time"]
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# =========================
# CAP: max 2 plays per game
# =========================
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

print("✅ Sims:", SIMS, "| SD_MARGIN:", SD_MARGIN, "| SD_TOTAL:", SD_TOTAL)
print("✅ Plays after cap:", len(card))

top = card.head(20).copy()
top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_final_card.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

✅ Sims: 50000 | SD_MARGIN: 14.5 | SD_TOTAL: 18.0
✅ Plays after cap: 13
                                                                                                      discord_text
              Cleveland Cavaliers at Brooklyn Nets\nBrooklyn Nets ML (+420) — 0.25u\nSim Win%: 25.3% | EV: 31.7%\n
                  Milwaukee Bucks at Chicago Bulls\nChicago Bulls ML (+136) — 0.44u\nSim Win%: 55.0% | EV: 29.7%\n
                 Milwaukee Bucks at Chicago Bulls\nChicago Bulls 3.5 (-110) — 0.51u\nSim Win%: 64.5% | EV: 23.1%\n
               Minnesota Timberwolves at Denver Nuggets\nUNDER 237.5 (-106) — 0.46u\nSim Win%: 62.5% | EV: 21.5%\n
                    Philadelphia 76ers at Boston Celtics\nOVER 221.5 (-106) — 0.45u\nSim Win%: 62.3% | EV: 21.0%\n
         Portland Trail Blazers at Atlanta Hawks\nAtlanta Hawks -6.0 (-108) — 0.43u\nSim Win%: 62.4% | EV: 20.1%\n
                Portland Trail Blazers at Atlanta Hawks\nUNDER 237.5 (-106) — 0.34u\nSim Win%: 59.7% | EV: 16.1%\n
         

In [ ]:
import numpy as np
import pandas as pd

# Uses your existing games_df that already has:
# mean_margin, mean_total, spread_home_line, spread_away_line, odds, etc
SIMS = 50000
SD_MARGIN = globals().get("SD_MARGIN", 14.5)
SD_TOTAL  = globals().get("SD_TOTAL", 18.0)

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# numeric safety
for c in ["home_ml","away_ml","spread_home_line","spread_away_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

# Spread cover: home covers its listed spread (home_line)
# home covers if margin + home_line > 0  <=> margin > -home_line
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]

df["p_home_win"] = (margins > 0).mean(axis=1)
df["p_over"]  = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"] = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half-Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def fmt_line(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# ML (home only)
# =========================
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# =========================
# SPREAD (FIXED LINE SIGN)
# =========================
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()

sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])

# ✅ Use the actual market line from feed (NO NEGATION)
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], sp["spread_away_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])

sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].apply(fmt_line) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# =========================
# TOTAL (best side)
# =========================
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    return x[["matchup","market","bet","units","p_win","ev","commence_time"]]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# =========================
# CAP: max 2 plays per game
# =========================
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

# =========================
# DISCORD TEXT
# =========================
top = card.head(20).copy()
top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print("✅ FIXED spread signs. Plays after cap:", len(top))
print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_FIXED_card.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

✅ FIXED spread signs. Plays after cap: 13
                                                                                                       discord_text
               Cleveland Cavaliers at Brooklyn Nets\nBrooklyn Nets ML (+420) — 0.25u\nSim Win%: 25.3% | EV: 31.7%\n
                   Milwaukee Bucks at Chicago Bulls\nChicago Bulls ML (+136) — 0.44u\nSim Win%: 55.0% | EV: 29.7%\n
                 Milwaukee Bucks at Chicago Bulls\nChicago Bulls +3.5 (-110) — 0.51u\nSim Win%: 64.5% | EV: 23.1%\n
                Minnesota Timberwolves at Denver Nuggets\nUNDER 237.5 (-106) — 0.46u\nSim Win%: 62.5% | EV: 21.5%\n
                     Philadelphia 76ers at Boston Celtics\nOVER 221.5 (-106) — 0.45u\nSim Win%: 62.3% | EV: 21.0%\n
            Portland Trail Blazers at Atlanta Hawks\nAtlanta Hawks -6 (-108) — 0.43u\nSim Win%: 62.4% | EV: 20.1%\n
                 Portland Trail Blazers at Atlanta Hawks\nUNDER 237.5 (-106) — 0.34u\nSim Win%: 59.7% | EV: 16.1%\n
                   Detroit Pis